# 06 — CNN–Transformer (Reduced-Electrode)

**Family:** CNN–Transformer &nbsp;•&nbsp; **Input:** raw signal, **5 channels only**

Same CNN + Transformer family as notebook 04, but restricted to a consumer-grade
5-electrode montage: **AF3, F3, F4, AF4, O1** — the frontal-asymmetry quartet
(a valence / arousal correlate) plus one occipital reference. This quantifies how
much accuracy is lost when going from a 14-channel research headset to a minimal
wearable layout.

## 1. Setup

In [ ]:
import os, sys, time, json
from functools import partial

import numpy as np
import torch

# make the `src` package importable when the notebook runs from notebooks/
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from src.evaluation import (
    set_seed, get_device, count_parameters,
    make_default_fit, default_predict,
    run_subject_dependent, run_loso, save_results, quick_report,
)
from src.data_loader import (
    build_subject_dependent_loaders, build_loso_loaders, load_subject, REDUCED_5_CHANNELS,
)

set_seed(42)
DEVICE = get_device()

PROCESSED_DIR = os.path.join(ROOT, "data", "processed")
SD_SPLITS     = os.path.join(ROOT, "data", "splits", "subject_dependent_splits.json")
LOSO_SPLITS   = os.path.join(ROOT, "data", "splits", "loso_splits.json")
RESULTS_DIR   = os.path.join(ROOT, "results")

# ----- experiment configuration -----
LABEL = "valence"     # main comparison target. Swap to "arousal" / "dominance" to reproduce.
QUICK = False         # True  -> a 3-subject / 3-fold smoke test with few epochs
                      # False -> the full benchmark used for the thesis table
print("device:", DEVICE, "| label:", LABEL, "| QUICK:", QUICK)

from src.models import cnn_transformer_reduced
MODEL_NAME = "CNN_Transformer_Reduced"
print("reduced montage:", REDUCED_5_CHANNELS)

## 2. Architecture

`src/models/cnn_transformer_reduced.py`: a 3-block 1-D temporal CNN stem
(downsamples time ×8) → sinusoidal positional encoding → 3 Transformer encoder
layers → additive temporal-attention pooling → linear head. Identical in spirit to
CIT-EmotionNet's global branch, adapted to raw multi-channel time series.

In [ ]:
# Instantiate once on a dummy spec just to print the graph + parameter count.
_dummy_meta = {"input_shape": (5, load_subject(PROCESSED_DIR, "subject_01").window_samples)}
_m = cnn_transformer_reduced.build({"n_classes": 2, "meta": _dummy_meta})
print(_m)
print(f"\ntrainable parameters: {count_parameters(_m):,}")
del _m

## Validation protocols

Both protocols are run through the shared harness in `src/evaluation.py` so the
numbers are directly comparable across all eight notebooks.

* **Subject-dependent** — for each of the 23 subjects a fresh model is trained on
  that subject's training split and tested on their held-out split
  (`data/splits/subject_dependent_splits.json`; whole film clips are kept on one
  side of the split to avoid window leakage). We report the mean ± std of the
  per-subject scores.
* **Subject-independent / LOSO** — 23 leave-one-subject-out folds
  (`data/splits/loso_splits.json`). Fold *k* trains on 22 subjects and tests on
  the 23rd; a 10% slice of the training subjects is used for early stopping.
  We report the mean ± std across the 23 folds.

The gap between the two is the central object of study in the thesis.

### 3. Subject-dependent protocol

In [ ]:
FEATURE  = "raw"
CHANNELS = REDUCED_5_CHANNELS
BATCH_SD = 64
EPOCHS_SD = 60
LR = 0.0007

sd_loader = partial(
    build_subject_dependent_loaders,
    processed_dir=PROCESSED_DIR, splits_path=SD_SPLITS,
    label=LABEL, feature=FEATURE, batch_size=BATCH_SD,
    channels=CHANNELS, normalize=True,
)
fit_sd = make_default_fit(epochs=(5 if QUICK else EPOCHS_SD), lr=LR)
sd_subjects = ["subject_01", "subject_02", "subject_03"] if QUICK else None

sd_res = run_subject_dependent(
    build_model=cnn_transformer_reduced.build,
    processed_dir=PROCESSED_DIR, splits_path=SD_SPLITS,
    loader_factory=sd_loader, fit_fn=fit_sd, predict_fn=default_predict,
    device=DEVICE, model_kwargs={"n_classes": 2, **{}},
    subjects=sd_subjects,
)
quick_report(MODEL_NAME, sd_res)

### 4. Subject-independent (LOSO) protocol

In [ ]:
BATCH_LOSO = 128
EPOCHS_LOSO = 50

loso_loader = partial(
    build_loso_loaders,
    processed_dir=PROCESSED_DIR, loso_path=LOSO_SPLITS,
    label=LABEL, feature=FEATURE, batch_size=BATCH_LOSO,
    channels=CHANNELS, normalize=True, val_frac=0.1,
)
fit_loso = make_default_fit(epochs=(5 if QUICK else EPOCHS_LOSO), lr=LR)
loso_folds = list(range(3)) if QUICK else None

loso_res = run_loso(
    build_model=cnn_transformer_reduced.build, loso_path=LOSO_SPLITS,
    loader_factory=loso_loader, fit_fn=fit_loso, predict_fn=default_predict,
    device=DEVICE, model_kwargs={"n_classes": 2, **{}},
    folds=loso_folds,
)
quick_report(MODEL_NAME, loso_res)

## 5. Save results

In [ ]:
save_results(
    MODEL_NAME,
    subject_dependent=sd_res, loso=loso_res,
    family="CNN-Transformer (5ch)", results_dir=RESULTS_DIR,
    config={"label": LABEL, "feature": FEATURE, "channels": CHANNELS,
             "batch_sd": BATCH_SD, "batch_loso": BATCH_LOSO,
             "epochs_sd": EPOCHS_SD, "epochs_loso": EPOCHS_LOSO, "lr": LR,
             "quick": QUICK},
)

## Notes for the methodology chapter

* The 5-channel subset is selected in `src/data_loader.REDUCED_5_CHANNELS`; change
  it there to test other minimal montages.
* Compare this row directly against notebook 04 (CIT-EmotionNet, 14ch DE) and
  notebook 01 (EEGNet, 14ch raw) for the "how few electrodes can we get away with"
  discussion.


## Notes for the methodology chapter

* Every metric here comes from `src/evaluation.compute_metrics` (accuracy, macro
  precision / recall / F1, confusion matrix) — no per-notebook metric code.
* Parameter count and wall-clock training time are logged to
  `results/{MODEL_NAME}_results.json` for the accuracy-vs-efficiency analysis.
* Set `QUICK = True` at the top to smoke-test the whole flow in a few minutes
  before committing to the full multi-hour run.
* `LABEL` switches the target between valence / arousal / dominance without any
  other change.